# ML Model Training: Model Promotion

This notebook mirrors the **Model Training** guide at [learnmlops.ops4life.com/guides/ml-model-training/](https://learnmlops.ops4life.com/guides/ml-model-training/).

Walk through the full model promotion workflow: find the best run in MLflow, evaluate it against quality thresholds, register it in the model registry, and transition it to Staging.

**What we'll cover:**
1. List experiments and runs
2. Define promotion criteria
3. Register and promote the model
4. Confirm the model version in the registry

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri('http://mlflow:5000')
client = MlflowClient()

## Step 1: List experiments and runs

In [ ]:
# Find the employee-attrition experiment
experiments = client.search_experiments()
print('Experiments:')
for e in experiments:
    print(f'  [{e.experiment_id}] {e.name}')

In [ ]:
# Get all runs from the attrition experiment sorted by ROC-AUC
try:
    exp = client.get_experiment_by_name('employee-attrition')
    if exp is None:
        print("No 'employee-attrition' experiment yet. Run train.ipynb first.")
    else:
        runs = client.search_runs(
            experiment_ids=[exp.experiment_id],
            order_by=['metrics.roc_auc DESC'],
            max_results=5,
        )
        print(f'Top {len(runs)} runs in employee-attrition:')
        for r in runs:
            auc = r.data.metrics.get('roc_auc', 'N/A')
            f1  = r.data.metrics.get('f1', 'N/A')
            print(f'  run_id={r.info.run_id[:8]}...  roc_auc={auc}  f1={f1}  status={r.info.status}')
        best_run = runs[0] if runs else None
except Exception as e:
    print(f'Error: {e}')
    best_run = None

## Step 2: Define promotion criteria

In [ ]:
def evaluate_and_promote(run_id, model_name, thresholds):
    """
    Evaluate a run against quality thresholds and promote it if it passes.

    Args:
        run_id:     MLflow run ID to evaluate
        model_name: Registered model name in the MLflow registry
        thresholds: Dict with keys min_roc_auc, min_f1

    Returns:
        model_version: The newly registered ModelVersion, or None if promotion failed
    """
    run = client.get_run(run_id)
    metrics = run.data.metrics

    roc_auc = metrics.get('roc_auc', 0)
    f1      = metrics.get('f1', 0)

    print(f'Evaluating run {run_id[:8]}...')
    print(f'  ROC-AUC: {roc_auc:.4f}  (threshold: >= {thresholds["min_roc_auc"]})')
    print(f'  F1:      {f1:.4f}  (threshold: >= {thresholds["min_f1"]})')

    if roc_auc < thresholds['min_roc_auc']:
        raise ValueError(f'ROC-AUC {roc_auc:.4f} below threshold {thresholds["min_roc_auc"]}')
    if f1 < thresholds['min_f1']:
        raise ValueError(f'F1 {f1:.4f} below threshold {thresholds["min_f1"]}')

    # Register the model
    model_uri = f'runs:/{run_id}/model'
    mv = mlflow.register_model(model_uri, model_name)
    print(f'\nRegistered: {model_name} v{mv.version}')

    # Transition to Staging
    client.transition_model_version_stage(
        name=model_name,
        version=mv.version,
        stage='Staging',
        archive_existing_versions=False,
    )
    print(f'Promoted to: Staging')
    return mv

The function applies two gates before promotion:
1. **Absolute thresholds** — hard minimum on ROC-AUC and F1
2. **Registry integration** — on passing, the run is registered as a new model version and transitioned to Staging

In a CI/CD pipeline this function would be called automatically after each training run.

## Step 3: Register and promote the model

In [ ]:
if best_run is not None:
    try:
        mv = evaluate_and_promote(
            run_id=best_run.info.run_id,
            model_name='attrition-classifier',
            thresholds={'min_roc_auc': 0.50, 'min_f1': 0.10},
        )
    except ValueError as e:
        print(f'Promotion blocked: {e}')
        mv = None
else:
    print('No runs found. Run train.ipynb first.')
    mv = None

## Step 4: Confirm the model version in the registry

In [ ]:
try:
    versions = client.get_latest_versions('attrition-classifier')
    print('Registry: attrition-classifier')
    for v in versions:
        print(f'  v{v.version}  stage={v.current_stage}  run={v.run_id[:8]}...')
except Exception as e:
    print(f'Registry lookup: {e}')

Open the MLflow UI at [mlflow.learnmlops.ops4life.com](https://mlflow.learnmlops.ops4life.com) and navigate to **Models → attrition-classifier** to see the registered versions and their stage transitions.

## Next Steps

- Monitor the model for data drift: `04-operations/data-drift-model-decay/drift_detection.ipynb`
- Full guide: [learnmlops.ops4life.com/guides/ml-model-training/](https://learnmlops.ops4life.com/guides/ml-model-training/)